In [1]:
pip install transformers

In [3]:
!pip install -q sentence-transformers pymupdf scikit-learn pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 67.6 MB/s eta 0:00:00


In [4]:
#upload file
from google.colab import files
uploaded=files.upload()

Saving Education.zip to Education.zip


In [5]:
#unzip file
!unzip Education.zip

Streaming output truncated to the last 5000 lines.
  inflating: Education/Resume_030001_Lori_Hart.pdf  
  inflating: Education/Resume_030002_Anthony_Moses.pdf  
  inflating: Education/Resume_030003_Edward_Shepherd.pdf  
  inflating: Education/Resume_030004_Nicholas_Potter.pdf  
  inflating: Education/Resume_030005_Donna_Watts.pdf  
  inflating: Education/Resume_030006_Jennifer_Oconnor.pdf  
  inflating: Education/Resume_030007_Antonio_Barnes.pdf  
  inflating: Education/Resume_030008_Alexander_Huang.pdf  
  inflating: Education/Resume_030009_Courtney_Clark.pdf  
  inflating: Education/Resume_030010_Samuel_George.pdf  
  inflating: Education/Resume_030011_Kyle_George.pdf  
  inflating: Education/Resume_030012_Jason_Moore.pdf  
  inflating: Education/Resume_030013_Christina_Mason.pdf  
  inflating: Education/Resume_030014_Kathryn_Alexander.pdf  
  inflating: Education/Resume_030015_Nathan_White.pdf  
  inflating: Education/Resume_030016_Ethan_Carlson.pdf  
  inflating: Education/Resume_0

In [6]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

In [7]:
model=SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

#Extracting text from pdfs


In [11]:
import fitz # PyMuPDF
import os
pdf_dir = 'Education'
pdf_texts = []
pdf_filenames = []

for filename in tqdm(os.listdir(pdf_dir), desc="Extracting text from PDFs"):
    if filename.endswith('.pdf'):
        filepath = os.path.join(pdf_dir, filename)
        try:
            doc = fitz.open(filepath)
            text = ""
            for page in doc:
                text += page.get_text()
            pdf_texts.append(text)
            pdf_filenames.append(filename)
            doc.close()
        except Exception as e:
            print(f"Error processing {filename}: {e}")

print(f"\nSuccessfully extracted text from {len(pdf_texts)} PDF files.")


Extracting text from PDFs: 100%|██████████| 5000/5000 [00:32<00:00, 155.54it/s]


Successfully extracted text from 5000 PDF files.


uploading job description file


In [12]:
#upload jd file
from google.colab import files
jds_uploaded=files.upload()

Saving 20_Job_Descriptions (2).pdf to 20_Job_Descriptions (2).pdf


#Extracting text from job description pdf


In [13]:
jd_filepath=list(jds_uploaded.keys())[0]
try:
  jd_doc=fitz.open(jd_filepath)
  jd_text=""
  for page in jd_doc:
    jd_text+=page.get_text()
  jd_doc.close()
  print(f"Successfully extracted text from '{jd_filepath}'.")
  print("\n--- Job Description Content (first 500 characters)---")
  print(jd_text[:500])
  print("---------------------------------------")
except Exception as e:
  print(f"Error processing {jd_filepath}: {e}")

Successfully extracted text from '20_Job_Descriptions (2).pdf'.

--- Job Description Content (first 500 characters)---
20 Professional Job Descriptions
Azure Data Engineer
Job Summary
We are seeking a skilled Azure Data Engineer to join our team and contribute to designing,
developing, and supporting business-critical solutions.
Key Responsibilities
• Design, develop, and maintain enterprise solutions.
• Collaborate with cross-functional teams and stakeholders.
• Troubleshoot issues and optimize performance.
• Follow best practices for quality, security, and compliance.
• Prepare technical documentation and repo
---------------------------------------


#Generating embeddings for job descriptio and resumes


In [15]:
jd_names=[]
jd_texts=[]
jd_pdf=fitz.open(
    '20_Job_Descriptions (2).pdf'
)
for page_num in range(
    len(jd_pdf)
):
  page=jd_pdf[page_num]
  text=page.get_text()
  title=text.split(
      "\n"
  )[0].strip()
  enhanced_text=(
      title + " "
  ) * 10 + text
  jd_names.append(
      title
  )
  jd_texts.append(
      enhanced_text
  )
print(
    "Total JDs:",
    len(jd_names)
)
print(jd_names)
print(os.listdir())

Total JDs: 20
['20 Professional Job Descriptions', 'Azure DevOps Engineer', 'Data Scientist', 'Data Analyst', 'Full Stack Developer', 'Python Developer', 'Machine Learning Engineer', 'Generative AI Engineer', 'Cloud Engineer', 'Site Reliability Engineer (SRE)', 'Business Analyst', 'SQL Developer', 'Power BI Developer', 'ETL Developer', 'Big Data Engineer', 'MLOps Engineer', 'Kubernetes Administrator', 'Cyber Security Analyst', 'Digital Marketing Specialist', 'Project Manager']
['.config', '20_Job_Descriptions (2).pdf', 'Education.zip', 'Education', 'sample_data']


In [16]:
resume_embeddings=model.encode(
    pdf_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)
print(
    resume_embeddings.shape
)

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

(5000, 384)


In [ ]:
#now save the embeddings
np.save(
    'resume_embeddings.npy',
    resume_embeddings
)

In [17]:
#save resume metadata
resume_names=pdf_filenames
resume_paths=[os.path.join(pdf_dir, filename) for filename in pdf_filenames]
pd.DataFrame({
    "resume_name":resume_names,
    "resume_path":resume_paths
}).to_csv(
    "resume_metadata.csv",
    index=False
)
print("Resume metadata saved to 'resume_metadata.csv")

Resume metadata saved to 'resume_metadata.csv


In [19]:
#generate embedding for jd
jd_embeddings=model.encode(
    jd_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)
print(f"Shape of job description embeddings: {jd_embeddings.shape}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Shape of job description embeddings: (20, 384)


In [20]:
#save job description embedings
np.save(
    'jd_embeddings.npy',
    jd_embeddings
)
print("Job description embeddings saved to 'jd_embeddings.npy")

Job description embeddings saved to 'jd_embeddings.npy


In [21]:
#calculate cosine similarity and find top matches
cosine_similarities=cosine_similarity(jd_embeddings,resume_embeddings)
print(f"shape of cosine similarity matrix: {cosine_similarities.shape}")

shape of cosine similarity matrix: (20, 5000)


In [23]:
#display top matching resumes for each job description
num_top_matches=20
results=[]
resume_metadata=pd.read_csv('resume_metadata.csv')
for i,jd_name in enumerate(jd_names):
  top_resume_indices=np.argsort(cosine_similarities[i])[::-1][:num_top_matches]
  for rank, idx in enumerate(top_resume_indices):
    resume_filename=resume_metadata.loc[idx, 'resume_name']
    similarity_score=cosine_similarities[i, idx]
    results.append({
        'Job Description':jd_name,
        'Rank':rank+1,
        'Resume_Filename':resume_filename,
        'Similarity_Score':similarity_score
    })
matching_results_df=pd.DataFrame(results)
print("Top Matching Resumes:")
print(matching_results_df)

Top Matching Resumes:
                      Job Description  ...  Similarity_Score
0    20 Professional Job Descriptions  ...          0.624507
1    20 Professional Job Descriptions  ...          0.624461
2    20 Professional Job Descriptions  ...          0.622648
3    20 Professional Job Descriptions  ...          0.622075
4    20 Professional Job Descriptions  ...          0.619636
..                                ...  ...               ...
395                   Project Manager  ...          0.462331
396                   Project Manager  ...          0.461837
397                   Project Manager  ...          0.461100
398                   Project Manager  ...          0.460047
399                   Project Manager  ...          0.459725

[400 rows x 4 columns]


In [24]:
matching_results_df.head(20)

,Job Description,Rank,Resume_Filename,Similarity_Score
0,20 Professional Job Descriptions,1,Resume_033394_Steven_Watkins.pdf,0.624507
1,20 Professional Job Descriptions,2,Resume_033541_Mark_Zhang.pdf,0.624461
2,20 Professional Job Descriptions,3,Resume_030521_Suzanne_Bowman.pdf,0.622648
3,20 Professional Job Descriptions,4,Resume_033028_Kaylee_Anderson.pdf,0.622075
4,20 Professional Job Descriptions,5,Resume_034049_Amanda_Walker.pdf,0.619636
5,20 Professional Job Descriptions,6,Resume_032194_Cindy_Ramirez.pdf,0.617421
6,20 Professional Job Descriptions,7,Resume_033927_Robert_Cline.pdf,0.615522
7,20 Professional Job Descriptions,8,Resume_033674_Edgar_Watts.pdf,0.607900
8,20 Professional Job Descriptions,9,Resume_033274_Shelly_Lewis.pdf,0.602250
9,20 Professional Job Descriptions,10,Resume_031631_Todd_Burgess.pdf,0.593534


In [25]:
matching_results_df.to_csv('top_matching_resumes.csv', index=False)
print("Top matching resumes saved to 'top_matching_resumes.csv'")

Top matching resumes saved to 'top_matching_resumes.csv'
